# Choice-Stability Stopping Study

**What this notebook produces.** An evaluation of a new *decision-stability*
stopping criterion for `VectorizedModel`: stop when **every** agent's chosen
theory (greedy `argmax` of its credence) has been unchanged for the last `W`
steps — equivalently, no agent has crossed the 0.5 decision boundary in the last
`W` steps. It targets the false-convergence / theory-flip failure mode of the
tolerance rule (see `STOPPING_CONDITION_ANALYSIS.md` §3.1). Companion to
`CHOICE_STABILITY_STOPPING_PLAN.md`.

Three studies, mirroring the Study-2 parameter-search notebook:

- **Study 1 — Grid.** For `uncertainty × n_experiments × network`, run each seed
  **once** (native stop at the largest window, recording flips) and derive the
  stop step + truth share for **every** window `W ∈ {100,250,500,1000}` offline.
- **Study 1 — Analysis.** OLS testing **H1′** (steps driven by window,
  uncertainty, n_experiments) and **H2′** (truth share driven by uncertainty and
  ~invariant to window — the headline prediction).
- **Study 2 — Comparison.** Choice-stability vs tolerance stopping on shared
  seeds: stop step, truth share, and **post-stop flip rate** after resuming.

Set `SMOKE_TEST = True` for a fast Restart-and-Run-All; set it `False` on Colab
for the full grid.

# Setup

In [ ]:
# ── Environment switch + master parameters (single source of truth) ──────────
# RUNNING_LOCALLY=True  → run on a laptop from the repo (no clone/pip/Drive).
# RUNNING_LOCALLY=False → Colab: force-fresh clone, pip install, mount Drive.
import os, sys, subprocess, shutil
from pathlib import Path

RUNNING_LOCALLY = False
SMOKE_TEST      = True          # True → tiny grid + short runs (fast R&RA)
MASTER_SEED     = 20260717

# Tiered compute budgets (smoke → laptop → cloud); see NOTEBOOK_WRITING_SKILL §14.
# The grid + comparison checkpoint to Drive after every combo/criterion and skip
# work already saved, so a disconnect resumes instead of restarting from zero.
CS_N_RUNS   = 5 if SMOKE_TEST else (50 if RUNNING_LOCALLY else 250)
CS_MAX_STEPS = 3_000 if SMOKE_TEST else 100_000

# The choice-stability windows are DERIVED OFFLINE from one record-once run each.
CS_WINDOWS = [100, 250, 500, 1000]
if SMOKE_TEST:
    CS_WINDOWS = [50, 100, 200, 400]   # windows the short smoke runs can reach

print(f"RUNNING_LOCALLY={RUNNING_LOCALLY}  SMOKE_TEST={SMOKE_TEST}")
print(f"CS_N_RUNS={CS_N_RUNS}  CS_MAX_STEPS={CS_MAX_STEPS}  CS_WINDOWS={CS_WINDOWS}")

In [ ]:
# Force-fresh clone + deps (Colab only). Uses subprocess/os.chdir (not %/!) so
# the RUNNING_LOCALLY guard actually holds — see NOTEBOOK_WRITING_SKILL §5–6.
if not RUNNING_LOCALLY:
    if os.path.exists('e_network_inequality'):
        shutil.rmtree('e_network_inequality')
    subprocess.run(['git', 'clone', '-b', 'main',
                    'https://github.com/IgnacioOQ/e_network_inequality'], check=True)
    subprocess.run(['pip', 'install', '-q', 'dill'], check=True)
    os.chdir('e_network_inequality')

# Locate the project root (the inner dir containing model/) and put it on sys.path.
if RUNNING_LOCALLY:
    # This notebook lives in model/convergence_analysis/stopping_condition/.
    PROJECT_ROOT = Path.cwd()
    while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'model').is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
# Explicit imports (no wildcards from external libs).
import itertools, pickle, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from functools import partial
from multiprocessing import Pool, cpu_count
from numpy.random import SeedSequence
from tqdm.auto import tqdm

from model.vectorized_model import VectorizedModel
from model.vectorized_simulation_functions import run_vectorized_simulation_with_params

num_cores = cpu_count()
print(f"Available CPU cores: {num_cores}")

In [ ]:
# Output directory — one path constant per role, created once.
if RUNNING_LOCALLY:
    RESULTS_DIR = Path('model/convergence_analysis/stopping_condition')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = Path('/content/drive/My Drive/Colab Projects/Data Driven ABMs/'
                       'Data Sets/choice_stability_study')
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Results dir:", RESULTS_DIR)

def save_fig(name, dpi=150):
    plt.savefig(RESULTS_DIR / name, dpi=dpi, bbox_inches='tight')
    plt.show()

def save_csv(df, name):
    path = RESULTS_DIR / name
    df.to_csv(path, index=False)
    print(f"  Saved: {path} ({len(df)} rows)")
    return path

In [ ]:
def offline_stop(flip_history, window, max_steps):
    """Derive (stop_step, truth_share, stabilized) for a window from a
    record_choice_flips history [(step, truth_share), ...] — a baseline entry
    at run start plus one per step where >=1 agent flipped. Truth share is
    constant between flips, so the FIRST inter-event gap of length >= window
    fixes the stop. If none exists, the run did not stabilize within max_steps.

    NOTE the record-once run stops natively at the LARGEST window, so its history
    contains every flip needed to derive all smaller windows (proven equivalent
    to per-window native runs in test_stopping_conditions.py)."""
    steps = [s for s, _ in flip_history] + [max_steps]
    shares = [t for _, t in flip_history]
    for i in range(len(flip_history)):
        if steps[i + 1] - steps[i] >= window:
            return steps[i] + window, shares[i], True
    return max_steps, (shares[-1] if shares else np.nan), False

# Study 0: Load Networks

In [ ]:
def load_indexed(path):
    with open(path, 'rb') as f:
        G = pickle.load(f)
    G = nx.relabel_nodes(G, {n: i for i, n in enumerate(G.nodes())})
    print(f"  {os.path.basename(path)}: {G.number_of_nodes()} nodes, "
          f"{G.number_of_edges()} edges")
    return G

print("Loading networks:")
G_pud     = load_indexed('./networks/citation_data/pud_network.pkl')
G_tobacco = load_indexed('./networks/citation_data/tobacco_network.pkl')

NETWORKS = [("PUD", "pud", G_pud), ("Tobacco", "tobacco", G_tobacco)]

# Study 1: Choice-Stability Grid (record-once)

Grid over `uncertainty × n_experiments`. Each seed is run **once** with native
`choice_stability_stopping` at the largest window plus `record_choice_flips`;
every window is then derived offline (`offline_stop`). The window axis therefore
costs no extra simulation. Records `steps`, `truth_share`, `stabilized` per
(run, window).

In [ ]:
# ── Study 1 — Config ─────────────────────────────────────────────────────────
CS_UNCERTAINTIES = [0.0001, 0.001, 0.005, 0.01]
CS_N_EXPERIMENTS = [100, 500, 1000]
if SMOKE_TEST:
    CS_UNCERTAINTIES = [0.001, 0.01]
    CS_N_EXPERIMENTS = [100, 500]
print("uncertainties:", CS_UNCERTAINTIES)
print("n_experiments:", CS_N_EXPERIMENTS)
print("windows (offline):", CS_WINDOWS)
print("runs/combo:", CS_N_RUNS, " max_steps:", CS_MAX_STEPS)

In [ ]:
def run_choice_stability_grid(network, network_label, output_prefix,
                              master_seed=MASTER_SEED):
    # Checkpoint/resume: the per-combo results are written to the SAME CSV after
    # every combo, so a disconnect only loses the in-flight combo. On re-run,
    # combos already in the checkpoint are skipped (and a fully-done network is
    # skipped entirely). Per-combo seeds are spawned by combo index, so skipping
    # earlier combos does not shift the seeds of later ones.
    csv_path = RESULTS_DIR / f"{output_prefix}_choice_stability.csv"
    combos = list(itertools.product(CS_N_EXPERIMENTS, CS_UNCERTAINTIES))
    W_max = max(CS_WINDOWS)
    expected_per_combo = CS_N_RUNS * len(CS_WINDOWS)

    if csv_path.exists():
        done_df = pd.read_csv(csv_path)
        print(f"  [{network_label}] resuming from checkpoint ({len(done_df)} rows)")
    else:
        done_df = pd.DataFrame(columns=["n_experiments", "uncertainty", "window",
                                        "run", "steps", "truth_share", "stabilized"])

    def _combo_done(df, n_exp, unc):
        if df.empty:
            return False
        m = (df["n_experiments"] == n_exp) & (np.isclose(df["uncertainty"], unc))
        return int(m.sum()) >= expected_per_combo

    combo_seqs = SeedSequence(master_seed).spawn(len(combos))   # one per combo
    rows = done_df.to_dict("records")
    t_start = time.time()
    print(f"  [{network_label}] {len(combos)} combos × {CS_N_RUNS} runs = "
          f"{len(combos)*CS_N_RUNS:,} simulations (all windows derived offline)")

    with tqdm(list(enumerate(combos)), desc=f"[{network_label}] combos",
              unit="combo") as pbar:
        for idx, (n_exp, unc) in pbar:
            if _combo_done(done_df, n_exp, unc):
                pbar.set_postfix(skip=f"n_exp={n_exp} unc={unc}")
                continue
            child_seeds = [int(s.generate_state(1)[0])
                           for s in combo_seqs[idx].spawn(CS_N_RUNS)]
            param_dicts = [{"network": network, "n_experiments": n_exp,
                            "uncertainty": unc, "seed": cs} for cs in child_seeds]

            # Native stop at the largest window + record flips → one run per seed.
            wrapper = partial(
                run_vectorized_simulation_with_params,
                tolerance_stopping=False,
                choice_stability_stopping=True,
                choice_stability_window=W_max,
                record_choice_flips=True,
                number_of_steps=CS_MAX_STEPS,
                show_bar=False, agent_type="beta",
            )
            t0 = time.time()
            with Pool(num_cores) as pool:
                results = list(tqdm(pool.imap_unordered(wrapper, param_dicts),
                                    total=CS_N_RUNS, leave=False,
                                    desc=f"  n_exp={n_exp} unc={unc:.4f}"))
            for run_idx, r in enumerate(results):
                flips = r["choice_flip_history"]
                for W in CS_WINDOWS:
                    stop, tshare, stab = offline_stop(flips, W, CS_MAX_STEPS)
                    rows.append({"n_experiments": n_exp, "uncertainty": unc,
                                 "window": W, "run": run_idx, "steps": stop,
                                 "truth_share": tshare, "stabilized": int(stab)})
            # Checkpoint after every combo (atomic — only complete combos persist).
            pd.DataFrame(rows).to_csv(csv_path, index=False)
            pbar.set_postfix(last=f"{time.time()-t0:.1f}s", rows=f"{len(rows):,}")

    print(f"  [{network_label}] Done — {len(rows)} rows in "
          f"{(time.time()-t_start)/60:.1f} min")

    df = pd.DataFrame(rows)
    save_csv(df, f"{output_prefix}_choice_stability.csv")
    summary = (df.groupby(["n_experiments", "uncertainty", "window"])
                 .agg(mean_steps=("steps", "mean"), std_steps=("steps", "std"),
                      mean_truth=("truth_share", "mean"),
                      std_truth=("truth_share", "std"),
                      stabilized_rate=("stabilized", "mean")).reset_index())
    save_csv(summary, f"{output_prefix}_choice_stability_summary.csv")
    _plot_grid(df, network_label, output_prefix)
    return df

In [ ]:
def _plot_grid(df, network_label, output_prefix):
    # (a) Stop-time distribution per window (log scale) — false-convergence lens.
    plt.figure(figsize=(8, 5))
    data = [df[df.window == W]["steps"].values for W in CS_WINDOWS]
    plt.boxplot(data, labels=[str(W) for W in CS_WINDOWS])
    plt.yscale('log'); plt.xlabel("Window W (flip-free steps)")
    plt.ylabel("Stop step (log)")
    plt.title(f"Choice-stability stop-time vs window — {network_label}")
    plt.grid(True, alpha=0.3, which='both')
    save_fig(f"{output_prefix}_choice_stability_boxplot.png")

    # (b) Window sensitivity: mean steps (should rise) and mean truth share
    #     (should stay ~flat — the H2' visual), faceted by n_experiments.
    for metric, ylab, logy in [("steps", "Mean stop step (log)", True),
                               ("truth_share", "Mean truth share", False)]:
        fig, axes = plt.subplots(1, len(CS_N_EXPERIMENTS), sharey=True,
                                 figsize=(5 * len(CS_N_EXPERIMENTS), 4.5))
        axes = np.atleast_1d(axes)
        colors = plt.cm.viridis(np.linspace(0, 0.9, len(CS_UNCERTAINTIES)))
        for ax, n_exp in zip(axes, CS_N_EXPERIMENTS):
            sub = df[df.n_experiments == n_exp]
            for unc, c in zip(CS_UNCERTAINTIES, colors):
                means = [sub[(sub.uncertainty == unc) & (sub.window == W)][metric].mean()
                         for W in CS_WINDOWS]
                ax.plot([str(W) for W in CS_WINDOWS], means, 'o-', color=c,
                        label=f"unc={unc}", linewidth=1.8, markersize=5)
            if logy: ax.set_yscale('log')
            else:    ax.set_ylim(0, 1)
            ax.set_title(f"n_experiments={n_exp}"); ax.set_xlabel("Window W")
            ax.grid(True, alpha=0.3)
            if ax is axes[0]: ax.set_ylabel(ylab)
        axes[-1].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
        fig.suptitle(f"{ylab.split(' (')[0]} vs window — {network_label}")
        plt.tight_layout()
        save_fig(f"{output_prefix}_choice_stability_{metric}_lines.png")

## PUD Network

In [ ]:
print("=== Choice-Stability Grid: PUD Network ===")
df_pud = run_choice_stability_grid(G_pud, "PUD", "pud")

## Tobacco Network

In [ ]:
print("=== Choice-Stability Grid: Tobacco Network ===")
df_tobacco = run_choice_stability_grid(G_tobacco, "Tobacco", "tobacco")

# Study 1: Data Analysis — H1′ / H2′

Log₁₀-transformed predictors `log_window`, `log_uncertainty`,
`log_n_experiments`. **H1′:** steps driven by window (+ uncertainty,
n_experiments). **H2′:** truth share driven by uncertainty and ~invariant to
window (its standardized coefficient / Cohen's f² for `log_window` should be
near zero).

In [ ]:
from utils.data_analysis_utils import (
    compute_correlations, compute_vif,
    run_ols, regression_diagnostics, plot_f2_comparison,
)

In [ ]:
def analyze_choice_stability(network_label, output_prefix):
    df = pd.read_csv(RESULTS_DIR / f"{output_prefix}_choice_stability.csv")
    df['log_window']        = np.log10(df['window'])
    df['log_uncertainty']   = np.log10(df['uncertainty'])
    df['log_n_experiments'] = np.log10(df['n_experiments'])
    predictors = ['log_window', 'log_uncertainty', 'log_n_experiments']

    print(f"\n{'='*60}\n  Choice-Stability Analysis: {network_label}  "
          f"(n={len(df)})\n{'='*60}\n")

    print("── Multicollinearity: Pearson correlations ──")
    compute_correlations(df, predictors)
    print("\n── Multicollinearity: Variance Inflation Factors ──")
    display(compute_vif(df, predictors))

    print("\n── Regression: Steps to Stabilization  (H1′) ──")
    model_steps = run_ols(df, predictors, dependent='steps',
                          label='Steps to Stabilization',
                          network_label=network_label)
    regression_diagnostics(model_steps,
                           title=f"Diagnostics: Steps — {network_label}")

    print("\n── Regression: Truth Share at Stabilization  (H2′) ──")
    model_truth = run_ols(df, predictors, dependent='truth_share',
                          label='Truth Share', network_label=network_label)
    regression_diagnostics(model_truth,
                           title=f"Diagnostics: Truth Share — {network_label}")

    plot_f2_comparison(
        models={'Steps to Stabilization': model_steps,
                'Truth Share': model_truth},
        predictors=predictors,
        title=f"Cohen's f² — {network_label} (expect log_window ~0 for Truth Share)")

## PUD Network

In [ ]:
print("=== Choice-Stability Data Analysis: PUD Network ===")
analyze_choice_stability("PUD", "pud")

## Tobacco Network

In [ ]:
print("=== Choice-Stability Data Analysis: Tobacco Network ===")
analyze_choice_stability("Tobacco", "tobacco")

# Study 2: Comparison vs Tolerance Stopping + Post-Stop Drift

On **shared seeds**, compare tolerance stopping (default `tol=5e-3`) against
choice-stability stopping for each window `W`, at a representative parameter
point. For each stopped state we resume `K` extra steps and measure the
**post-stop flip fraction** (agents that cross the 0.5 boundary after the stop).
Hypothesis (RQ4): choice-stability yields a far lower post-stop flip rate than
tolerance stopping.

In [ ]:
# ── Study 2 — Config ─────────────────────────────────────────────────────────
CMP_UNCERTAINTY   = 0.001
CMP_N_EXPERIMENTS = 500 if not SMOKE_TEST else 100
CMP_TOLERANCE     = 5e-3
CMP_N_RUNS        = 5 if SMOKE_TEST else (30 if RUNNING_LOCALLY else 100)
CMP_CONTINUATION  = 200 if SMOKE_TEST else 10_000    # K steps resumed after stop
print(f"unc={CMP_UNCERTAINTY} n_exp={CMP_N_EXPERIMENTS} tol={CMP_TOLERANCE} "
      f"runs={CMP_N_RUNS} K={CMP_CONTINUATION}")

In [ ]:
def _believes_truth(cred):
    return cred[:, 1] > cred[:, 0]

def _run_and_resume(network, unc, n_exp, seed, stop_kwargs, K, max_steps):
    """Run one model to its stop, then resume K steps from the stopped state and
    report stop step, truth share, and the fraction of agents that flipped."""
    m = VectorizedModel(network=network, n_experiments=n_exp, agent_type="beta",
                        uncertainty=unc, seed=seed, seeded=True, **stop_kwargs)
    m.run_simulation(number_of_steps=max_steps, show_bar=False)
    stop_step = m.n_steps
    truth = float(np.mean(_believes_truth(m.credences)))
    believes_at_stop = _believes_truth(m.credences)
    ab_stop, cred_stop = m.alphas_betas.copy(), m.credences.copy()

    # Resume from the exact stopped state (fixed steps).
    r = VectorizedModel(network=network, n_experiments=n_exp, agent_type="beta",
                        uncertainty=unc, tolerance_stopping=False,
                        tstep_stopping=True, seeded=False)
    r.alphas_betas, r.credences = ab_stop, cred_stop
    r.run_simulation(number_of_steps=K, show_bar=False)
    flip_frac = float(np.mean(_believes_truth(r.credences) != believes_at_stop))
    return stop_step, truth, flip_frac

def run_comparison(network, network_label, output_prefix, master_seed=MASTER_SEED):
    csv_path = RESULTS_DIR / f"{output_prefix}_stopping_comparison.csv"
    seeds = [int(s.generate_state(1)[0])
             for s in SeedSequence(master_seed + 1).spawn(CMP_N_RUNS)]
    criteria = [("tolerance", dict(tolerance=CMP_TOLERANCE, tolerance_stopping=True))]
    criteria += [(f"choice_W{W}",
                  dict(tolerance_stopping=False, choice_stability_stopping=True,
                       choice_stability_window=W)) for W in CS_WINDOWS]

    # Checkpoint/resume by criterion (same pattern as the grid): a disconnect
    # only loses the in-flight criterion; on re-run, completed criteria skip.
    if csv_path.exists():
        done_df = pd.read_csv(csv_path)
        print(f"  [{network_label}] resuming comparison from checkpoint "
              f"({len(done_df)} rows)")
    else:
        done_df = pd.DataFrame(columns=["criterion", "seed", "stop_step",
                                        "truth_share", "post_stop_flip_frac"])
    rows = done_df.to_dict("records")
    for crit, kw in tqdm(criteria, desc=f"[{network_label}] criteria"):
        if (not done_df.empty) and int((done_df["criterion"] == crit).sum()) >= CMP_N_RUNS:
            continue
        for seed in seeds:
            stop, truth, flip = _run_and_resume(
                network, CMP_UNCERTAINTY, CMP_N_EXPERIMENTS, seed, kw,
                CMP_CONTINUATION, CS_MAX_STEPS)
            rows.append({"criterion": crit, "seed": seed, "stop_step": stop,
                         "truth_share": truth, "post_stop_flip_frac": flip})
        pd.DataFrame(rows).to_csv(csv_path, index=False)   # checkpoint per criterion
    df = pd.DataFrame(rows)
    save_csv(df, f"{output_prefix}_stopping_comparison.csv")

    summ = (df.groupby("criterion")
              .agg(mean_stop=("stop_step", "mean"),
                   mean_truth=("truth_share", "mean"),
                   mean_flip=("post_stop_flip_frac", "mean"),
                   any_flip_rate=("post_stop_flip_frac", lambda s: float(np.mean(s > 0))))
              .reindex([c for c, _ in criteria]).reset_index())
    print(summ.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    order = [c for c, _ in criteria]
    axes[0].bar(order, summ["mean_stop"], color="steelblue")
    axes[0].set_yscale('log'); axes[0].set_ylabel("Mean stop step (log)")
    axes[0].set_title(f"Stop step — {network_label}")
    axes[0].tick_params(axis='x', rotation=45)
    axes[1].bar(order, summ["mean_flip"], color="indianred")
    axes[1].set_ylabel(f"Mean post-stop flip fraction (K={CMP_CONTINUATION})")
    axes[1].set_title(f"Post-stop drift — {network_label}")
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    save_fig(f"{output_prefix}_stopping_comparison.png")
    return df

## PUD Network

In [ ]:
print("=== Stopping Comparison: PUD Network ===")
cmp_pud = run_comparison(G_pud, "PUD", "pud")

## Tobacco Network

In [ ]:
print("=== Stopping Comparison: Tobacco Network ===")
cmp_tobacco = run_comparison(G_tobacco, "Tobacco", "tobacco")

# Disconnect from Runtime

In [ ]:
# Release the Colab runtime last (no-op locally). Print before disconnecting.
if not RUNNING_LOCALLY:
    from datetime import datetime
    import pytz
    from IPython.display import Javascript
    stamp = datetime.now(pytz.timezone('America/New_York')).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"✅ Finished at {stamp} — disconnecting runtime.")
    display(Javascript('google.colab.kernel.disconnect()'))
else:
    print("Local run complete (runtime not disconnected).")